***
# Homework 5: Functional Programming

**Course:** STAT 606 - Computing in Data Science and Statistics SP24

**Name:** Shrivats Sudhir

**NetID:** ssudhir2

**Email:** ssudhir2@wisc.edu

**Collaborators:** Samuel Merten.

**Date:** February 30, 2024
***

## 1.) Iterators and Generators (4 points $\approx$ 30 minutes)

**In this exercise, you’ll get some practice working with iterators and generators. Note: in this problem, the word enumerate is meant in the sense of returning elements, not in the sense of the Python function enumerate. So, if I say that an iterator enumerates a sequence $a_0, a_1, a_2, \dots$ , I mean that these are the elements that it returns upon calls to the `__next__` method, not that it returns pairs `(i, a_i)` like the enumerate function.**

**Define a class `Fibo` of iterators that enumerate the Fibonacci numbers. For the purposes of this problem, the Fibonacci sequence begins $0, 1, 1, 2, 3, \dots$ , with the $n$-th Fibonacci number $F_n$ given by the recursive formula $F_n = F_{n−1} + F_{n−2}$. Your solution should not make use of any function aside from addition (i.e., you should not need to use the function `fibo()` defined in lecture a few weeks ago). Your class should support, at a minimum, an initialization method, a `__iter__` method (so that we can get an iterator) and a `__next__` method. Note: there is an especially simple solution to this problem that can be expressed in just a few lines using tuple assignment.**

In [14]:
class Fibo:

    def __init__(self): 
        self.F0, self.F1 = 0, 1 # F0 = 0 and F1 = 1 to generate fibonacci sequence

    def __iter__(self):
        return self # return the object itself
    
    def __next__(self):
        curr = self.F0 # current fibonacci number
        self.F0, self.F1 = self.F1, self.F0 + self.F1 # Fn = Fn-1 + Fn-2, updates F0 and F1
        return curr # return current fibonacci number

### Testing class `Fibo`

In [15]:
F = Fibo()
for _ in range(10):
    print(next( F ))

0
1
1
2
3
5
8
13
21
34


**We can generalize the Fibonacci sequence by following the same recursive procedure $F_n = F_{n−1} +F_{n−2}$, but using a different choice of initial two values for $F_0$ and $F_1$. For example, if we take $F_0 = 2$ and $F_1 = 1$, then we obtain the Lucas numbers, which are closely related to the Fibonacci numbers. Define a class `GenFibo` of iterators that enumerate generalized Fibonacci numbers. Your class should inherit from the Fibo class defined in the previous subproblem. The initialization method for the GenFibo class should take two optional arguments that specify the values of `F0` and `F1`, in that order, and their values should default so that `F = GenFibo()` results in an iterator that enumerates the same sequence as if you had called `F = Fibo()`. That is, `GenFibo()` should produce an iterator over the Fibonacci numbers.**

In [16]:
class GenFibo( Fibo ):

    def __init__(self, F0 = 0, F1 = 1): # default F0 = 0 and F1 = 1 to generate fibonacci sequence
        super().__init__() # calls the __init__ method of the parent class Fibo

        self.F0, self.F1 = F0, F1 # F0, F1 depends on user input

    def __iter__(self):
        return self # returns the object itself
    
    def __next__(self):
        curr = self.F0 # current fibonacci number
        self.F0, self.F1 = self.F1, self.F0 + self.F1 # Fn = Fn-1 + Fn-2, updates F0 and F1
        return curr # return current fibonacci number

### Testing class `GenFibo`

In [17]:
GF = GenFibo(F0 = 2, F1 = 1)
for _ in range(10):
    print(next( GF ))

2
1
3
4
7
11
18
29
47
76


**Define a generator `squared_primes` that enumerates the squares of the prime numbers. Recall that a prime number is any integer $p > 1$ whose only divisors are $p$ and $1$**

**Note: you may use the function `is_prime` that we defined in lecture (or some- thing similar to it), but such solutions will not receive full credit, as there is a more graceful solution that avoids declaring a separate function or method for directly checking primality. Hint: consider a pattern similar to the one seen in lecture using the any and/or all functions.**

In [18]:
def squared_primes():
    
    curr = 2 # first prime number
    yield curr**2

    while True:
        curr += 1 # check the next number after curr = 2.
        # 1. Check if curr is divisible by any number between 2 and sqrt(curr).
        # 2. If curr is not divisible by any number between 2 and sqrt(curr), then yield square of curr.
        # 3. Else, check if the next number is prime.
        while all( (curr % i) for i in range(2, int(curr**0.5) + 1) ) == False:
            curr += 1 # if curr is not prime, check the next number
        yield curr**2 # if curr is prime, yield curr**2


### Testing generator `squared_primes()`

In [19]:
SP = squared_primes()

for i in range(10):
    print(next(SP))

4
9
25
49
121
169
289
361
529
841


**This one is good practice for coding interview questions. The Ulam numbers are a sequence $u_1, u_2, u_3, \dots$ of positive integers, defined in the following way: $u_1 = 1$, and $u_2 = 2$. For all $n > 2$, $u_n$ is the smallest integer that is expressible as a sum of two distinct terms from earlier in the sequence in exactly one way. Define a generator `ulam` that enumerates the Ulam numbers. Hint: it will be helpful to try and break this problem into smaller, simpler subproblems. In particular, you may find it helpful to write a function that takes a list of integers `t` and one additional integer `u`, and determines whether or not `u` is expressible as a sum of two distinct elements of `t` in exactly one way.**

In [36]:
def ulam():
    curr_ulams = [1,2] # first two ulam numbers
    idx = 0 # index of the first ulam number
    yield curr_ulams[idx] # yield first ulam number
    yield curr_ulams[idx + 1] # yield second ulam number
    while True:

        # generate all possible sums of two distinct ulam numbers
        # i loops through all ulam numbers currently found.
        # j loops through all ulam numbers currently found after i.
        sums = [curr_ulams[i] + curr_ulams[j] for i in range(len(curr_ulams)) 
                                              for j in range(i + 1, len(curr_ulams))]
        
        # i > curr_ulams[-1] ensures that the next ulam number is greater than the last ulam number found.
        # sums.count(i) == 1 ensures that the next ulam number is unique.
        # next_ulam finds the next ulam number.
        next_ulam = min( i for i in sums if i > curr_ulams[-1] and sums.count(i) == 1 )

        curr_ulams.append(next_ulam)
        yield next_ulam # yield next ulam number

In [37]:
ulam_gen = ulam()
for i in range(10):
    print(next(ulam_gen))

1
2
3
4
6
8
11
13
16
18


## 2.) List Comprehensions and Generator Expressions (4 points, spent $\approx$ 20 minutes)

**In this exercise you’ll write a few simple list comprehensions and generator expressions. Again in this problem we use the term *enumerate* to mean that a list comprehension or generator expression returns certain elements, rather than in the sense of the Python function `enumerate`**

**Write a list comprehension that enumerates the sequence $2^n−1$ for $n = 0, 1, 2, 3, \dots , 20$. For ease of grading, please assign this list comprehension to a variable called `pow2minus1`**

In [38]:
pow2minus1 = [ (2**i - 1) for i in range(21) ]
print(pow2minus1)

[0, 1, 3, 7, 15, 31, 63, 127, 255, 511, 1023, 2047, 4095, 8191, 16383, 32767, 65535, 131071, 262143, 524287, 1048575]


**The *Lazy Caterer’s sequence* is a sequence of numbers that counts, for each $n = 0, 1, 2, \dots$, the largest number of pieces that can be cut from a disk with at most $n$ cuts. The n-th number in this sequence is given by $p_n = (n^2 + n + 2)/2$, where $n = 0, 1, 2, \dots$. Write a generator expression that enumerates the Lazy Caterer’s sequence. For ease of grading, please assign this generator expression to a variable called `caterer`. Hint: you may find it useful to define a generator that enumerates the non-negative integers.**

In [39]:
# Used the generator from E09_func.ipynb
def non_negative_intgen():
    n = 0
    while True:
        yield n
        n += 1

non_negative = non_negative_intgen()
caterer = ( (n**2 + n + 2) // 2  for n in non_negative )

In [40]:
for i in range(10):
    print( next(caterer) )

1
2
4
7
11
16
22
29
37
46


**Write a generator expression that enumerates the tetrahedral numbers. The n-th tetrahedral number ($n = 1, 2, \dots$ ) is given by, $T_n = \binom{n+2}{3}$, where:**

$$ \binom{x}{y} = \frac{x!}{y!(x-y)!} $$

**For ease of grading, please assign this generator expression to a variable called `tetra`. Hint: you may find it useful to define a generator that enumerates the positive integers.**

In [41]:
# Used the factorial function from ssudhir2_hw1.ipynb
def factorial(k):
    # All inputs are as specified, i.e., k is a non-negative integer (no error-checking).
    if k == 0: # 0! = 1
        return 1 # base case for recursion       
    else: # k! = k * (k-1)!
        return k * factorial(k-1) # recursive step
    
# Used the generator from E09_func.ipynb
def positive_intgen(): 
    n = 1 # positive integers start from 1
    while True:
        yield n 
        n += 1

positive_int = positive_intgen()
tetra = ( (factorial(n+2) // (factorial(3) * factorial(n-1))) for n in positive_int)

In [26]:
for i in range(10):
    print( next(tetra) )

1
4
10
20
35
56
84
120
165
220


## 3.) Map, Filter and Reduce (3 points, spent $\approx$ 15 minutes)

**In this exercise, you’ll learn a bit about map, filter and reduce operations. We will revisit these operations in a few weeks when we discuss MapReduce and related frameworks in distributed computing. In this problem, I expect that you will use only the functions `map`, `filter` and functions from the `functools` and `itertools` modules, along with the `range` function (and similar list-related functions) and a sprinkling of `lambda` expressions**

In [7]:
from functools import reduce
from itertools import accumulate

**Write a one-line expression that computes the sum of the first 10 even square numbers (starting from 4). For ease of grading, please assign the output of this expression to a variable called `sum_of_even_squares`.**

In [4]:
sum_of_even_squares = reduce(lambda x,y : x + y, # 3. takes sum of all elements in a list, returns a single number
                             map(lambda x: x**2, # 2. takes squares of all elements in a list, returns a list
                             filter(lambda x: x % 2 == 0, range(21)))) # 1. finds all even numbers between 0 and 20, returns a list
print(sum_of_even_squares)

1540


**Write a one-line expression that computes the product of the first 13 primes. You may use (a modification of) the `squared_primes` generator that you defined above. For ease of grading, please assign the output of this expression to a variable called `product_of_primes`.**

In [5]:
product_of_primes = reduce(lambda x,y : x*y, # 2. takes product of all elements in a list, returns a single number
                           filter(lambda x: all( x % i for i in range(2, int(x**0.5) + 1) ) == True, range(2,42))) # 1. finds all prime numbers between 2 and 41, returns a list
print(product_of_primes)

304250263527210


**Write a one-line expression that computes the sum of the squares of the first 31 primes. You may use the `squared_primes` generator that you defined above. For ease of grading, please assign the output of this expression to a variable called `sum_of_squared_primes`.**

In [67]:
sum_of_squared_primes = reduce(lambda x,y : x + y, # 3. takes sum of all elements in a list, returns a single number
                               map(lambda x: x**2, # 2. takes squares of all elements in a list, returns a list
                               filter(lambda x: all( x % i for i in range(2, int(x**0.5) + 1) ) == True, range(2,130)))) # 1. finds all prime numbers between 2 and 129, returns a list
print(sum_of_squared_primes)

138834


**Write a one-line expression that computes a list of the first twenty harmonic numbers. Recall that the $n$-th harmonic number is given by $H_n = \sum_{k=1}^{n} 1/k$. For ease of
grading, please assign the output of this expression to a variable called `harmonics`.**

In [8]:
harmonics = list(accumulate( # 2. takes moving sum of all elements in a list, returns a list
                 map(lambda x: 1/x, range(1, 21)))) # 1. takes reciprocals of all elements in a list, returns a list
print(harmonics)

[1.0,
 1.5,
 1.8333333333333333,
 2.083333333333333,
 2.283333333333333,
 2.4499999999999997,
 2.5928571428571425,
 2.7178571428571425,
 2.8289682539682537,
 2.9289682539682538,
 3.0198773448773446,
 3.103210678210678,
 3.180133755133755,
 3.251562326562327,
 3.3182289932289937,
 3.3807289932289937,
 3.439552522640758,
 3.4951080781963135,
 3.547739657143682,
 3.597739657143682]

**For $n = 1, 2, 3, \dots$ , the $n$-th triangular number is given by:**

$$ T_n = \sum_{k=1}^{n} k = \binom{n+1}{2}$$

**Write a one-line expression that computes the geometric mean of the first 12 triangular numbers. Recall that the geometric mean of a collection of $n$ numbers $a_1, a_2, \dots , a_n$ is given by $\bigg(\prod_{i=1}^{n} a_i\bigg)^{1/n}$. For ease of grading, please assign the output of this expression to a variable called `tri_geom`.**

In [10]:
tri_geom = reduce(lambda x,y: x*y, # 2. takes product of all elements in a list, returns a single number, and then takes the 12th root of the product
                  list(accumulate(range(1, 13))))**(1/12) # 1. takes moving product of all elements in a list, returns a list
print(tri_geom)

17.318945452600932


## 4.) Fun with Polynomials (4 points, spent $\approx$ 10 minutes)

**In this exercise you’ll get a bit of experience writing higher-order functions.**

**Write a function `make_poly` that takes a list of numbers (ints and/or floats) coeffs as its only argument and returns a function `p`. The list coeffs encodes the coefficients of a polynomial, $p(x) = a_0 + a_1x + a_2x^2 + \dots + a_nx^n$, with $a_i$ given by `coeffs[i]`. The function `p` should take a single number (int or float) `x` as its argument, and return the value of the polynomial `p` evaluated at `x`. Your function should raise an appropriate error in the event that coeffs or one of its entries is not of the appropriate type.**

In [23]:
def make_poly(coeffs):

    if not isinstance(coeffs, (list, )):
        raise TypeError(f'coefficients {coeffs} must be of type list.') # raises error if coeffs is not a list
    elif all(isinstance(i, (int, float, )) for i in coeffs) == False:
        raise TypeError(f'all coefficients in {coeffs} must be of type int or float.') # raises error if coeffs contains elements that are not int or float

    def p(x):
        poly = coeffs[0] # leading coefficient a_0
        for i in range(1, len(coeffs)):
            poly += coeffs[i] * (x**i) # a_i * x^i for all 0 < i < len(coeffs)
        return poly
    return p

**Write a function `eval_poly` that takes two lists of numbers (ints and/or floats), `coeffs` and `args`. coeffs encodes the coefficients of polynomial `p`, and your function should return the list of numbers (ints and/or floats) representing the result of evaluating the polynomial `p` on each of the elements in `args`, in order of appearance. You should be able to express the solution to this problem in a single line (not including the function definition header and error checking, of course). Your function should make use of `make_poly` from the previous part to receive full credit. Your function should raise an appropriate error in the event that `coeffs`, `args` or one of their entries is not of the appropriate type**

In [27]:
def eval_poly(coeffs, args):

    if not isinstance(coeffs, (list, )):
        raise TypeError(f'coefficients {coeffs} must be of type list.') # raises error if coeffs is not a list
    elif all(isinstance(i, (int, float, )) for i in coeffs) == False:
        raise TypeError(f'all coefficients in {coeffs} must be of type int or float.') # raises error if coeffs contains elements that are not int or float
    
    if not isinstance(args, (list, )):
        raise TypeError(f'coefficients {args} must be of type list.') # raises error if args is not a list
    elif all(isinstance(i, (int, float, )) for i in args) == False:
        raise TypeError(f'all arguements in {args} must be of type int or float.') # raises error if args contains elements that are not int or float
    
    return list(map(lambda x: make_poly(coeffs)(x), args)) # returns a list of evaluations of the polynomial at the given args, in order of appearance